# 🎯 Targeted Feature Discovery & Density Audit
**Goal:** Identify the most information-dense OpenStreetMap (OSM) tags to optimize the Oracle's semantic search speed.

### 🧠 Scientific Rationale
In the Manhattan dataset, the POI (Point of Interest) DataFrame contains **1,000+ potential columns**. Performing a row-wise `.apply()` search across all columns is computationally expensive ($O(N \times M)$ complexity), leading to bottlenecks in batch labeling.

This notebook implements a **Density-First Audit** to:
1.  **Isolate String Data:** Filter out geometric and numerical noise to focus on semantic tags.
2.  **Quantify Information Density:** Rank columns by their "fill rate" (non-null percentage) to identify primary features (e.g., `amenity` vs. `description`).
3.  **Validate Keyword Sensitivity:** Track how specific navigational intents (like "post") are distributed across tags to ensure we don't lose **Recall** when we switch to **Targeted Vectorization**.

### 🛠️ Technical Objective
By identifying the top ~10 high-value columns, we can transition the `OracleEngine` from a "Global Search" to a **"Sniper Search"**, potentially increasing iteration speed by **20x–50x** without sacrificing grounding accuracy.

In [1]:
import pandas as pd

# Load your Manhattan data
df = pd.read_pickle(r'C:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_poi.pkl')

# 1. Find columns that actually have data (not all NaN)
populated_cols = df.columns[df.notna().any()].tolist()

# 2. Filter for 'Object' (String) columns only
string_cols = df[populated_cols].select_dtypes(include=['object']).columns.tolist()

# 3. Rank columns by "Information Density" (How many rows have a value)
density = df[string_cols].notna().mean().sort_values(ascending=False)

print("--- Top 10 Columns to Target ---")
print(density.head(10))

# 4. Check for 'Hidden' intent (e.g., searching for "Post")
example_query = "post"
hits = {}
for col in string_cols:
    count = df[col].str.contains(example_query, case=False, na=False).sum()
    if count > 0:
        hits[col] = count

print("\n--- Columns containing 'post' ---")
print(dict(sorted(hits.items(), key=lambda item: item[1], reverse=True)))

c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\.venv\Lib\site-packages\pandas\compat\pickle_compat.py:80: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)
C:\Users\adan\miniconda3\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


--- Top 10 Columns to Target ---
cellids             1.000000
unique_id           1.000000
osmid               1.000000
element_type        1.000000
centroid            1.000000
name                0.670957
amenity             0.556318
addr:street         0.433529
addr:housenumber    0.428953
addr:postcode       0.338100
dtype: float64

--- Columns containing 'post' ---
{'amenity': np.int64(308), 'operator': np.int64(170), 'operator:wikipedia': np.int64(166), 'brand:wikipedia': np.int64(87), 'brand': np.int64(55), 'name': np.int64(44), 'website': np.int64(8), 'wikipedia': np.int64(5), 'description': np.int64(5), 'alt_name': np.int64(3), 'shop': np.int64(2), 'fixme': np.int64(1), 'name:en': np.int64(1), 'image': np.int64(1), 'denomination': np.int64(1), 'information': np.int64(1), 'architect': np.int64(1), 'building:architecture': np.int64(1), 'post_office': np.int64(1)}


In [2]:
# Check for the specific landmarks that we normalized
target_queries = ["museum", "gallery", "park", "theater", "statue"]
expanded_hits = {}

for query in target_queries:
    for col in string_cols:
        count = df[col].str.contains(query, case=False, na=False).sum()
        if count > 0:
            expanded_hits[col] = expanded_hits.get(col, 0) + count

print("--- Vital Columns for Landmarks ---")
print(dict(sorted(expanded_hits.items(), key=lambda item: item[1], reverse=True)))

--- Vital Columns for Landmarks ---
{'amenity': np.int64(3179), 'name': np.int64(487), 'website': np.int64(315), 'leisure': np.int64(293), 'addr:street': np.int64(195), 'tourism': np.int64(112), 'wikipedia': np.int64(97), 'operator': np.int64(62), 'artwork_type': np.int64(58), 'vending': np.int64(44), 'cityracks.street': np.int64(36), 'description': np.int64(28), 'alt_name': np.int64(23), 'email': np.int64(18), 'building': np.int64(16), 'name:en': np.int64(15), 'official_name': np.int64(8), 'old_name': np.int64(8), 'branch': np.int64(8), 'note': np.int64(7), 'brand': np.int64(6), 'source': np.int64(5), 'inscription': np.int64(5), 'addr:housename': np.int64(5), 'memorial': np.int64(5), 'name:de': np.int64(4), 'image': np.int64(4), 'contact:instagram': np.int64(3), 'contact:twitter': np.int64(3), 'protection_title': np.int64(3), 'contact:facebook': np.int64(3), 'brand:wikipedia': np.int64(3), 'memorial:type': np.int64(3), 'inscription:url': np.int64(2), 'name:es': np.int64(1), 'opening_h